# Objetivo: Implementar un código que permita ORDENAR cada dirección (Sin eliminar ni ignorar direcciones iguales ya que esto solo indica que hay más de un solo paciente viviendo en esta dirección). La idea es ordenarlos por la **distancia más optima y cercana** para crear una *ruta optima que será utilizada para asginarle al doctor encargado de visitar al paciente.*

### Este es un ejemplo de un **'Traveling Salesman Problem' (Problema del Viajero)**

### Es un problema clásico de optimización matemática. La pregunta es simple: dado un conjunto de puntos (ciudades, direcciones, pacientes), ¿cuál es el orden de visita que minimiza la distancia total recorrida?

### En este caso: ***el "viajero" es el doctor, los "puntos" son las casas de los pacientes***

### **El problema tiene trampa.**
### Con 10 direcciones hay *3.6 millones de rutas posibles*. Con 100 direcciones, *el número supera los átomos del universo*. Por eso no se resuelve por fuerza bruta, sino con **algoritmos de aproximación.**

In [14]:
! pip install googlemaps
! pip install ortools
! pip install ortools scipy numpy pandas

In [15]:
from google.colab import drive
import googlemaps
import pandas as pd
import numpy as np
import time
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2, pywrapcp

# 0.2 - Importamos los datos con las direcciones

In [48]:
drive.mount('/content/drive', force_remount=True)
dataset = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Allevio/Batched PatientsAddresses.xlsx')
dataset

Mounted at /content/drive


,Encrypted Chart Number,Patient Address,Patient Zip Code,Appointment Date,Color
0,1,12526 OLD OAKS DR,77024-4940,2025-07-15,TEAL WEST
1,2,9552 REDWING LN,77316-2660,2026-01-28,YELLOW WEST
2,3,14323 HARTSHILL,77044-5067,2025-06-02,RED
3,4,1603 STONE MEADOWS LN,77094-3034,2025-10-15,TEAL WEST
4,5,8440 N SAM HOUSTON PKWY E,77396-2993,2025-07-07,LT BLUE
...,...,...,...,...,...
194,195,1315 Cherry Spring Dr,77038-2119,2026-03-18,GREY WEST
195,196,5306 ASKINS LANE,77016-1879,2025-12-11,RED
196,197,9550 FM 1488 RD ROOM 7,77354-4798,2026-03-16,LT GREEN NORTH
197,198,12910 Blossom Heath Rd,77429-2215,2026-03-23,ORANGE N.


# 0.3 - Limpiamos el dataset y hacemos correcciones

### Creamos una columna en el dataset que contenga la dirección completa del paciente puesto a que hace falta ciudad y estado (Houston, TX)

In [51]:
dataset['Full Address'] = dataset['Patient Address'] + ', Houston, TX ' + dataset['Patient Zip Code'].astype(str)
dataset

,Encrypted Chart Number,Patient Address,Patient Zip Code,Appointment Date,Color,Full Address
0,1,12526 OLD OAKS DR,77024-4940,2025-07-15,TEAL WEST,"12526 OLD OAKS DR, Houston, TX 77024-4940"
1,2,9552 REDWING LN,77316-2660,2026-01-28,YELLOW WEST,"9552 REDWING LN, Houston, TX 77316-2660"
2,3,14323 HARTSHILL,77044-5067,2025-06-02,RED,"14323 HARTSHILL, Houston, TX 77044-5067"
3,4,1603 STONE MEADOWS LN,77094-3034,2025-10-15,TEAL WEST,"1603 STONE MEADOWS LN, Houston, TX 77094-3034"
4,5,8440 N SAM HOUSTON PKWY E,77396-2993,2025-07-07,LT BLUE,"8440 N SAM HOUSTON PKWY E, Houston, TX 77396-2993"
...,...,...,...,...,...,...
194,195,1315 Cherry Spring Dr,77038-2119,2026-03-18,GREY WEST,"1315 Cherry Spring Dr, Houston, TX 77038-2119"
195,196,5306 ASKINS LANE,77016-1879,2025-12-11,RED,"5306 ASKINS LANE, Houston, TX 77016-1879"
196,197,9550 FM 1488 RD ROOM 7,77354-4798,2026-03-16,LT GREEN NORTH,"9550 FM 1488 RD ROOM 7, Houston, TX 77354-4798"
197,198,12910 Blossom Heath Rd,77429-2215,2026-03-23,ORANGE N.,"12910 Blossom Heath Rd, Houston, TX 77429-2215"


In [52]:
# Nos aseguramos que no haya pacientes con más de una cita registrada
dataset = dataset.sort_values('Appointment Date', ascending = False)
dataset = dataset.drop_duplicates(subset = 'Encrypted Chart Number', keep = 'first')
print(len(dataset))

199


# 0.4 - Creamos dos columnas para la geocodificación

In [53]:
dataset['Longitude'] = None
dataset['Latitude'] = None

In [54]:
dataset

,Encrypted Chart Number,Patient Address,Patient Zip Code,Appointment Date,Color,Full Address,Longitude,Latitude
54,55,4215 BRANDMERE WAY ST,77066-3600,2026-03-31,PINK,"4215 BRANDMERE WAY ST, Houston, TX 77066-3600",None,None
157,158,4903 BROWNFIELDS CT,77066-4711,2026-03-31,PINK,"4903 BROWNFIELDS CT, Houston, TX 77066-4711",None,None
111,112,7450 WILLOW CHASE BLVD,77070-5867,2026-03-30,ORANGE N.,"7450 WILLOW CHASE BLVD, Houston, TX 77070-5867",None,None
102,103,918 SPRING LAKES HAVEN,77373-8468,2026-03-26,LT BLUE,"918 SPRING LAKES HAVEN, Houston, TX 77373-8468",None,None
114,115,23202 BAYLEAF DR,77373-6202,2026-03-26,LT BLUE,"23202 BAYLEAF DR, Houston, TX 77373-6202",None,None
...,...,...,...,...,...,...,...,...
39,40,4407 PANTHER CREEK DR,77381-2400,2025-04-29,PURPLE,"4407 PANTHER CREEK DR, Houston, TX 77381-2400",None,None
45,46,501 NEWPORT BLVD,77573-4309,2025-04-28,TEAL EAST,"501 NEWPORT BLVD, Houston, TX 77573-4309",None,None
14,15,7979 N ELDRIDGE PKWY LOT 845,77041-1209,2025-04-25,ORANGE S.,"7979 N ELDRIDGE PKWY LOT 845, Houston, TX 7704...",None,None
48,49,19511 GLENWOODCANYON LN,77433-2725,2025-04-25,ORANGE SOUTH & NORTH,"19511 GLENWOODCANYON LN, Houston, TX 77433-2725",None,None


# 1 - Inicializamos API Google Maps

Whiteflag

In [39]:
# # Inicializamos cliente:
# gmaps = googlemaps.Client(key = '')

# def geocodeGoogle(address):
#     try:
#         result = gmaps.geocode(address)
#         if result:
#             loc = result[0]['geometry']['location']
#             return loc['lat'], loc['lng']
#     except Exception as e:
#         print(f"Error: {address} : {e}")
#     return None, None

# # Geocodificar y cachear:
# results = []
# for addr in dataset ['Full Address']:
#   lat, lng = geocodeGoogle(addr)
#   results.append({
#     'Encrypted Chart Number': dataset.loc[dataset['Full Address'] == addr, 'Encrypted Chart Number'].values[0],
#     'Full Address': addr,
#     'Latitude': lat,
#     'Longitude': lng
# })

# # Guardar caché:
# GeoCache = pd.DataFrame(results)
# GeoCache.to_excel('GeoCache.xlsx', index = False)
# GeoCache = GeoCache.drop_duplicates(subset='Full Address')

# # Merge al dataset original:
# dataset = dataset.drop(columns = ['Latitude', 'Longitude']).merge(GeoCache, on = 'Full Address', how = 'left')
# print(dataset[['Full Address', 'Latitude', 'Longitude']].head(10))
# print ('NaN restantes:', dataset['Latitude'].isna().sum())

# Verificaciones de resultados

In [28]:
# # Pacientes únicos
# print(dataset['Encrypted Chart Number'].nunique())
# # Direcciones únicas
# print(dataset['Full Address'].nunique())

In [29]:
# print(dataset['Full Address'].duplicated().sum()) # Verificación de que el paso 'merge' no añada filas de más'
# print(len(dataset))
# dataset

# 2 - Agrupar por dirección

## El TSP opera sobre puntos geográficos. Si dos pacientes habitan en una misma casa son un solo punto, no dos.

In [55]:
print(dataset.columns.tolist())
# Observamos que el cache duplicó la columna 'Encrypted Chart Number'.

['Encrypted Chart Number', 'Patient Address', 'Patient Zip Code', 'Appointment Date', 'Color', 'Full Address', 'Longitude', 'Latitude']


# Whiteflag

In [31]:
# # Limpiamos de nuevo
# dataset = dataset.drop(columns=['Encrypted Chart Number_y'])
# dataset = dataset.rename(columns={'Encrypted Chart Number_x': 'Encrypted Chart Number'})

In [65]:
# Agrupamos para evitar visitar la misma dirección dos veces.
addressGroups = dataset.groupby('Full Address').agg(
    patients = ('Encrypted Chart Number', list),
    lat = ('Latitude', 'first'),
    lng = ('Longitude', 'first'),
    color = ('Color', 'first')
).reset_index()

In [66]:
print(len(addressGroups))  # Vemos una diferencia de SEIS direcciones respecto al dataset inicial
# Esto índica que de todos los pacientes, solo 193 viven en una dirección única. Y que 6 de los 199 totales, comparten dirección con otro paciente.

193


In [ ]:
addressGroups.to_excel('Batched_Address_Groups.xlsx', index = False)
addressGroups
# Cada fila es una dirección única. 'patients' es la posición de la lista de Encrypted Chart Numbers que viven ahí. El doctor llega una vez y atiende a todos.
# Se exporta el dataframe a Excel para observar mejor la columna de 'patients'. Encontramos en las filas 65, 96, 115 y 151 los ID / charts de <1 paciente viviendo en una (1) dirección

,Full Address,patients,lat,lng,color
0,"101 LAKESIDE DR, Houston, TX 77356-9031",[66],None,None,YELLOW WEST
1,"10123 BRINWOOD DR, Houston, TX 77043-4304",[26],None,None,ORANGE S.
2,"10231 IVYRIDGE RD, Houston, TX 77043-4211",[108],None,None,ORANGE S.
3,"10318 MEADOW LAKE LN, Houston, TX 77042-2916",[163],None,None,TEAL WEST
4,"10507 WESTGATE DR, Houston, TX 77385-9566",[127],None,None,PURPLE
...,...,...,...,...,...
188,"9552 REDWING LN, Houston, TX 77316-2660",[2],None,None,YELLOW WEST
189,"9603 SAN CARLOS ST, Houston, TX 77013-3827",[96],None,None,RED
190,"9625 BAYOU BROOK ST, Houston, TX 77063-1058",[131],None,None,TEAL WEST
191,"9711 KNIGHTS DR, Houston, TX 77065-4337",[142],None,None,ORANGE N.


## Importamos el nuevo dataset sobre el que trabajaremos / latitud y longitud guardadas en caché

In [71]:
addressGroups = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Allevio/Batched addressGroups.xlsx')
addressGroups

,Full Address,patients,lat,lng,color
0,"101 LAKESIDE DR, Houston, TX 77356",[66],30.368849,-95.599916,YELLOW WEST
1,"10123 BRINWOOD DR, Houston, TX 77043",[26],29.794900,-95.546188,ORANGE S.
2,"10231 IVYRIDGE RD, Houston, TX 77043",[108],29.796916,-95.551111,ORANGE S.
3,"10318 MEADOW LAKE LN, Houston, TX 77042",[163],29.742884,-95.556557,TEAL WEST
4,"10507 WESTGATE DR, Houston, TX 77385",[127],30.164914,-95.437551,PURPLE
...,...,...,...,...,...
188,"9552 REDWING LN, Houston, TX 77316",[2],30.321230,-95.666004,YELLOW WEST
189,"9603 SAN CARLOS ST, Houston, TX 77013",[96],29.797123,-95.258674,RED
190,"9625 BAYOU BROOK ST, Houston, TX 77063",[131],29.748192,-95.538098,TEAL WEST
191,"9711 KNIGHTS DR, Houston, TX 77065",[142],29.913885,-95.602172,ORANGE N.


# 3 - Creamos una función (haversine) donde se aplique la ecuación de la distancia Haversine

In [72]:
def haversine (lat1, lng1, lat2, lng2):
  R = 6371000 # Radio de la tierra aproximado (en metros)
  phi1, phi2 = np.radians(lat1), np.radians(lat2) # Definimos latitudes EN RADIANTES como 'phi1' y 'phi2'
  dphi = np.radians (lat2 - lat1) # # Definimos LA DIFERENCIA ENTRE 'lat1' y 'lat2' para calcular la distancia angular. (Diferencia de latitud)
  dlambda = np.radians (lng2 - lng1) # Definimos LA DIFERENCIA ENTRE 'lng2' y 'lng1' para calcular la distancia angular. (Diferencia de longitud)
  a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2 # 'Valor del haversino': Obtiene el corazón de la fórmula. Mide la separción angular entre 2 puntos
  return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a)) # Convierte el ángulo alojado en la variable 'a' a distancia en metros.

In [75]:
# Unificamos las variantes de cada área de 'color':
colorMapping = {'ORANGE N.' : 'ORANGE',
  'ORANGE S.' : 'ORANGE',
  'ORANGE SOUTH & NORTH' : 'ORANGE',
  'GREY EAST' : 'GREY',
  'GREY WEST' : 'GREY',
  'GREY EAST & WEST' : 'GREY',
}

addressGroups['color'] = addressGroups['color'].replace(colorMapping)
addressGroups

,Full Address,patients,lat,lng,color
0,"101 LAKESIDE DR, Houston, TX 77356",[66],30.368849,-95.599916,YELLOW WEST
1,"10123 BRINWOOD DR, Houston, TX 77043",[26],29.794900,-95.546188,ORANGE
2,"10231 IVYRIDGE RD, Houston, TX 77043",[108],29.796916,-95.551111,ORANGE
3,"10318 MEADOW LAKE LN, Houston, TX 77042",[163],29.742884,-95.556557,TEAL WEST
4,"10507 WESTGATE DR, Houston, TX 77385",[127],30.164914,-95.437551,PURPLE
...,...,...,...,...,...
188,"9552 REDWING LN, Houston, TX 77316",[2],30.321230,-95.666004,YELLOW WEST
189,"9603 SAN CARLOS ST, Houston, TX 77013",[96],29.797123,-95.258674,RED
190,"9625 BAYOU BROOK ST, Houston, TX 77063",[131],29.748192,-95.538098,TEAL WEST
191,"9711 KNIGHTS DR, Houston, TX 77065",[142],29.913885,-95.602172,ORANGE


In [76]:
addressGroups.to_excel('addressGroups.xlsx', index = False)
addressGroups

,Full Address,patients,lat,lng,color
0,"101 LAKESIDE DR, Houston, TX 77356",[66],30.368849,-95.599916,YELLOW WEST
1,"10123 BRINWOOD DR, Houston, TX 77043",[26],29.794900,-95.546188,ORANGE
2,"10231 IVYRIDGE RD, Houston, TX 77043",[108],29.796916,-95.551111,ORANGE
3,"10318 MEADOW LAKE LN, Houston, TX 77042",[163],29.742884,-95.556557,TEAL WEST
4,"10507 WESTGATE DR, Houston, TX 77385",[127],30.164914,-95.437551,PURPLE
...,...,...,...,...,...
188,"9552 REDWING LN, Houston, TX 77316",[2],30.321230,-95.666004,YELLOW WEST
189,"9603 SAN CARLOS ST, Houston, TX 77013",[96],29.797123,-95.258674,RED
190,"9625 BAYOU BROOK ST, Houston, TX 77063",[131],29.748192,-95.538098,TEAL WEST
191,"9711 KNIGHTS DR, Houston, TX 77065",[142],29.913885,-95.602172,ORANGE


# 3.2 - Creamos una función (solveTSP)

In [77]:
def solveTSP(zone_df):
  """Función que calcula distancia Haversine""" # docstring
  coords = zone_df[['lat', 'lng']].values
  n = len(coords)

  # Construimos una tabla de distancias NxN entre todas las direcciones
  dist_matrix = np.zeros((n, n), dtype = int)
  for i in range (n):
    for j in range (n):
      dist_matrix[i][j] = int(haversine(*coords[i], *coords[j]))
  # OR-Tools : 1 doctor, empieza en índice 0
  manager = pywrapcp.RoutingIndexManager(n, 1, 0) # n = cantidad de direcciones, 1 = cantidad de doctores, 0 = índice de inicio (primera dirección de la zona)
  routing = pywrapcp.RoutingModel(manager) # Crea el modelo de optimización. "Motor" para resolver el TSP. Recibe la variable 'manager' como configuración.

  # Definimos una función que exprese ¿Cuánto cuesta ir de 'i' a 'j'? (Ir de una dirección a otra)
  def distanceCallback (i, j):
    return dist_matrix[manager.IndexToNode(i)][manager.IndexToNode(j)]
  cb = routing.RegisterTransitCallback(distanceCallback) # Guarda un ID de la función creada 'distanceCallback'
  routing.SetArcCostEvaluatorOfAllVehicles(cb) # Indica al doctor usar 'cb' para evaluar cada tramo. 'Arc' = Tramo entre 2 direcciones.

  params = pywrapcp.DefaultRoutingSearchParameters()
  params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
  params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
  params.time_limit.seconds = 30

  solution = routing.SolveWithParameters(params)

  # Extraemos el orden de visitas optimizado:

  order, index = [], routing.Start(0)
  while not routing.IsEnd(index):
    order.append(manager.IndexToNode(index))
    index = solution.Value(routing.NextVar(index))
  return zone_df.iloc[order].reset_index(drop = True)

## Corremos la función 'TSP' por cada zona de color

In [79]:
optimizedZones = {}
for color, group in addressGroups.groupby('color'):
  print(f"Optimizando {color} : {len(group)} direcciones")
  optimizedZones[color] = solveTSP(group)
# Cada zona se optimiza independientemente.

Optimizando DK BLUE : 10 direcciones
Optimizando DK GREEN CENTRAL : 2 direcciones
Optimizando DK GREEN SOUTH : 14 direcciones
Optimizando GREY : 7 direcciones
Optimizando KATY : 7 direcciones
Optimizando LT BLUE : 11 direcciones
Optimizando LT GREEN NORTH : 7 direcciones
Optimizando Livingston : 2 direcciones
Optimizando ORANGE : 24 direcciones
Optimizando PINK : 9 direcciones
Optimizando PURPLE : 9 direcciones
Optimizando RED : 9 direcciones
Optimizando TEAL CENTRAL : 19 direcciones
Optimizando TEAL EAST : 8 direcciones
Optimizando TEAL WEST : 19 direcciones
Optimizando Trinity : 1 direcciones
Optimizando YELLOW CENTRAL : 2 direcciones
Optimizando YELLOW EAST : 15 direcciones
Optimizando YELLOW NORTH : 10 direcciones
Optimizando YELLOW WEST : 8 direcciones


In [80]:
# Confirmo y chequeo este paciente puntual (114) ya que presentaba un error al no cambiar su subgrupo de área. Chequeo 'color':
addressGroups.loc[114]

,114
Full Address,"3114 ROGERS ST, Houston, TX 77022"
patients,[70]
lat,29.815912
lng,-95.385665
color,GREY


# 4 - Reconstruimos el dataset en orden optimizado

In [81]:
final_parts = []
for Color, zone_df in optimizedZones.items():
  zone_df = zone_df.copy()
  zone_df ['route_order'] = range(1, len(zone_df)+1)
  final_parts.append(zone_df)

optimizedDoc = pd.concat(final_parts, ignore_index = False)

# Exportamos a archivo en Excel.

with pd.ExcelWriter('Rutas_Optimizadas.xlsx', engine = 'openpyxl') as writer:
  # Hoja general:
  optimizedDoc.to_excel(writer, sheet_name = 'Todas las zonas', index = False)

  # Por zona:
  for color, zone_df in optimizedZones.items():
    zone_df = zone_df.copy()
    zone_df['route_order'] = range(1, len(zone_df) + 1)
    sheet_name = Color [:31] # Excel límita los nombres a 31 caracteres
    zone_df.to_excel(writer, sheet_name = sheet_name, index = False)

print('Exportado: Rutas_Optimizadas.xlsx')

Exportado: Rutas_Optimizadas.xlsx


## Confirmamos que se hayan guardado todas las zonas contenidas inicialmente

In [82]:
print(list(optimizedZones.keys()))
print(len(optimizedZones))

['DK BLUE', 'DK GREEN CENTRAL', 'DK GREEN SOUTH', 'GREY', 'KATY', 'LT BLUE', 'LT GREEN NORTH', 'Livingston', 'ORANGE', 'PINK', 'PURPLE', 'RED', 'TEAL CENTRAL', 'TEAL EAST', 'TEAL WEST', 'Trinity', 'YELLOW CENTRAL', 'YELLOW EAST', 'YELLOW NORTH', 'YELLOW WEST']
20


In [84]:
print(addressGroups['color'].unique())
print(len(addressGroups))

['YELLOW WEST' 'ORANGE' 'TEAL WEST' 'PURPLE' 'RED' 'YELLOW NORTH'
 'YELLOW EAST' 'LT BLUE' 'TEAL CENTRAL' 'DK BLUE' 'GREY' 'YELLOW CENTRAL'
 'PINK' 'TEAL EAST' 'DK GREEN SOUTH' 'Trinity' 'KATY' 'LT GREEN NORTH'
 'Livingston' 'DK GREEN CENTRAL']
